# Flying & singing zebra finches - Birdpark

Data from {cite:t}`ruttimann2025birdpark` ([paper](https://peerj.com/articles/20203), [zenodo](https://zenodo.org/records/13144875)), a multimodal dataset of zebra finch groups with synchronized video, microphone arrays, and backpack-mounted vibration transducer (accelerometers).

The code below shows how one can convert that existing dataset into the `Trials.nc` format. The sampling rate of the vibration transducer is very high (24kHz), so the raw `vibration` trace loads slowly; plot the downsampled `vibration_env` envelope instead (built below, together with candidate changepoints on its bursts), or use the **Downsample** option next to **Load**.

<img src="../docs/source/_static/media/birdpark1.png" width="1200">

Left: GUI screenshot, Right: Adapted from {cite:t}`ruttimann2025birdpark`, Fig. 2C

In [1]:
import zipfile
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import requests
import xarray as xr
from audioio import write_audio

import ethograph as eto
from ethograph.features.changepoints import find_nearest_turning_points_binary
from ethograph.features.energy import bandpass_envelope
from ethograph.io.pairing import pair_media

### Download dataset

You can download the entire dataset from [here](https://zenodo.org/records/13144875) or use the code below. I only tested the `copExpBP08` recording. If problems arise, the ReadMe [here](https://zenodo.org/records/13144875) is very helpful.

In [3]:
data_folder

WindowsPath('c:/Users/aksel/Documents/Code/ethograph/data/birdpark')

In [4]:
try:
    _here = Path(__vsc_ipynb_file__).parent
except NameError:
    _here = Path().resolve()

data_folder = _here.parent / "data" / "birdpark"
data_folder.mkdir(parents=True, exist_ok=True)

response = requests.get("https://zenodo.org/api/records/13144875")
data = response.json()

# Download dataset if not already present
if not (data_folder / "copExpBP08" / "BP_2021-05-25_08-12-51_655154_0380000.mp4").exists():
    for file in data["files"]:
        if file["checksum"] == "md5:32d1ae6049556c803f68b6d354c952ca":
            print(f"Checksum matches: {file['key']}")
            output_path = data_folder / file["key"]
            r = requests.get(file["links"]["self"], stream=True)
            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            if output_path.suffix == ".zip":
                with zipfile.ZipFile(output_path, "r") as zip_ref:
                    zip_ref.extractall(data_folder)
            break

### Build NWB alignment and dataset

In [ ]:
# Recording: copExpBP08, session BP_2021-05-25_08-12-51_655154_0380000
fps = 47.6837158203125
audio_sr = 24414.0625  # audio and accelerometer sampling rate (from dataset metadata)

recording_folder = data_folder / "copExpBP08"
h5_path = recording_folder / "BP_2021-05-25_08-12-51_655154_0380000.h5"
video_path = recording_folder / "BP_2021-05-25_08-12-51_655154_0380000.mp4"
audio_path = video_path.with_suffix(".wav")
nc_path = video_path.with_suffix(".nc")

# Read H5 file
with h5py.File(h5_path, "r") as f1:
    radioSignals = f1["/radioSignals"][()]  # accelerometer (one row per channel)
    daqSignals = f1["/daqSignals"][()]  # microphone channels

# Create .wav file from microphone channels
write_audio(audio_path, daqSignals.T, audio_sr)

# ─── Build session table ───
# trial=1 matches the ID assigned when loading a plain Dataset as a single-trial TrialTree
# The table names each file by its full path; pair_media stores the basename per
# trial and the path per stream, so the GUI finds the media without being told
# the folder (and a folder setting still overrides it on another machine).
session_table = pd.DataFrame(
    {
        "trial": [1],
        "video_0": [str(video_path)],
        "audio_0": [str(audio_path)],
    }
)

nwb_path = recording_folder / ".ethograph" / "alignment.nwb"
pair_media(
    trial_table=session_table,
    stream_rates={"video": float(fps), "audio": float(audio_sr)},
    output_path=nwb_path,
)

# ─── Build xarray dataset ───
time_coords = np.arange(radioSignals.shape[1]) / audio_sr

ds = xr.Dataset(
    data_vars={
        "vibration": xr.DataArray(
            radioSignals.T,
            dims=["time", "individual"],
        ),
    },
    coords={
        "time": time_coords,
        "individual": ["male (red radio)", "female (yellow radio)"],  # specific to copExpBP08
    },
    attrs={
        "fps": fps,
        "audio_sr": audio_sr,
    },
)

In [ ]:
# ─── Amplitude envelope + changepoints ───
# The raw 24 kHz trace is slow to plot and sits on a large DC offset (gravity +
# sensor bias), so what you actually want to look at and snap labels to is a
# bandpassed, rectified, smoothed envelope at a low rate. The envelope has its
# own (coarser) clock, so it gets its own time coord (`time_env`): any coord
# containing "time" is recognised by the GUI.
env_band = (10.0, 5000.0)  # drop the DC offset, keep body + song vibration
env_cutoff = 20.0  # smoothing of the rectified signal; must be < env_rate / 2
env_rate = 200.0

# Radio dropouts are stored as the sentinel -1e6. Show them as gaps in the raw
# trace, and fill them with the channel median before filtering so the step
# does not ring through the envelope.
ds["vibration"] = ds.vibration.where(ds.vibration > -999_999)

envelopes = []
for ind in ds.individual.values:
    raw = ds.vibration.sel(individual=ind)
    filled = raw.fillna(raw.median()).values
    env_time, env = bandpass_envelope(filled, audio_sr, band=env_band, env_rate=env_rate, cutoff=env_cutoff)
    envelopes.append(env)

ds["vibration_env"] = xr.DataArray(
    np.stack(envelopes, axis=1),
    dims=["time_env", "individual"],
    coords={"time_env": env_time},
)

# Candidate onsets/offsets: the flanks of each envelope burst, i.e. the nearest
# near-flat, near-baseline point on either side of a peak (`max_value` keeps the
# flanks off the burst itself). Thresholds are relative to the loud tail of the
# envelope; tune by eye in the GUI.
env_scale = float(np.nanpercentile(ds.vibration_env.values, 99))
ds = eto.add_changepoints_to_ds(
    ds=ds,
    target_feature="vibration_env",
    changepoint_name="turning_points",
    changepoint_func=find_nearest_turning_points_binary,
    threshold=0.02 * env_scale,  # |gradient| per sample counted as flat
    max_value=0.1 * env_scale,  # a flank must sit near baseline
    prominence=0.2 * env_scale,
    distance=int(0.05 * env_rate),  # ≥ 50 ms between bursts
)

ds.to_netcdf(nc_path)
print(f"Saved to {nc_path}")

Saved to c:\Users\aksel\Documents\Code\ethograph\data\birdpark\copExpBP08\BP_2021-05-25_08-12-51_655154_0380000.nc


In [10]:
ds.sel(individual='female (yellow radio)')

<xarray.Dataset> Size: 166MB
Dimensions:                       (time: 10240000, individuals: 2,
                                   time_env: 83935)
Coordinates:
  * time                          (time) float64 82MB 0.0 4.096e-05 ... 419.4
  * time_env                      (time_env) float64 671kB 0.0 ... 419.4
    individual                    <U21 84B 'female (yellow radio)'
Dimensions without coordinates: individuals
Data variables:
    vibration                     (time, individuals) float32 82MB 6.44e+05 ....
    vibration_env                 (time_env, individuals) float64 1MB 94.01 ....
    vibration_env_turning_points  (individuals, time_env) int8 168kB 1 0 ... 0 1
Attributes:
    fps:       47.6837158203125
    audio_sr:  24414.0625

In [ ]:
ds  # Inspect

<xarray.Dataset> Size: 166MB
Dimensions:                       (time: 10240000, individuals: 2,
                                   time_env: 83935)
Coordinates:
  * time                          (time) float64 82MB 0.0 4.096e-05 ... 419.4
  * individuals                   (individuals) <U21 168B 'male (red radio)' ...
  * time_env                      (time_env) float64 671kB 0.0 ... 419.4
Data variables:
    vibration                     (time, individuals) float32 82MB 6.44e+05 ....
    vibration_env                 (time_env, individuals) float64 1MB 94.01 ....
    vibration_env_turning_points  (individuals, time_env) int8 168kB 1 0 ... 0 1
Attributes:
    fps:       47.6837158203125
    audio_sr:  24414.0625

#### Decent for segmentation

```python
voc.segment.meansquared(
    <data>,  # set by GUI
    <sr>,    # set by GUI
    threshold=15000,
    min_dur=0.003,
    min_silent_dur=0.0001,
    freq_cutoffs=(500, 10000),
    smooth_win=0.32,
    scale=True,
    scale_val=32768,
)
```